In [3]:
import pandas as pd # import for data reading and manipulation
import re 
from pathlib import Path

%store -r MIMIC_DIR
%store -r cohort

In [4]:
# COMORBIDITY FLAGS
all_df = pd.read_csv("/home/emma/data/physionet.org/files/mimiciv/3.1/hosp/diagnoses_icd.csv", dtype = {"icd_code": str}) #manual approach
cohort_df = all_df[all_df.hadm_id.isin(cohort.hadm_id)]

def has_code(df, prefixes):
    pat = "^(" + "|".join(prefixes) + ")"
    return df.icd_code.str.match(pat)

comorbid_flags = cohort_df.groupby("hadm_id").agg( # comorbidity factors
    has_diabetes = ("icd_code", lambda x: x.str.match(r"^(250|E1[01234])").any()),
    has_renal_disease = ("icd_code", lambda x: x.str.match(r"^(585|N18)").any()),
    has_liver_disease = ("icd_code", lambda x: x.str.match(r"^(571|K7[0-7])").any()),
    has_copd = ("icd_code", lambda x: x.str.match(r"^(490|491|492|496|J4[0-4])").any()),
    has_substance_use_disorder = ("icd_code", lambda x: x.str.match(r"^(304|305|F1[1-9])").any()),   
).reset_index()
%store comorbid_flags

Stored 'comorbid_flags' (DataFrame)
